# Model Serving with a REST API

In production, a model does not sit in a Jupyter notebook waiting for you to call `model.predict()`. Other services—mobile apps, dashboards, backend jobs—need to reach it over HTTP. This notebook builds a FastAPI service around a trained sklearn model, writes it to disk, and then simulates calling it from Python.

**Learning objectives**
1. Explain why HTTP is the standard interface for deployed models.
2. Define a Pydantic request/response schema and understand what it validates.
3. Implement `/predict` and `/health` endpoints that load a model at startup.
4. Test the predict logic in-process without running a live server.


## 1  Why HTTP? The problem with `model.predict()` in production

A trained model lives as a Python object in RAM. Other teams write in JavaScript, Go, or Java; mobile apps run on iOS. None of them can import your Python object. An HTTP API solves this: any language can send a JSON request and receive a JSON response. The model stays in one place; everyone else calls it over the network.


## 2  Install dependencies


In [ ]:
# Run this once; restart kernel if packages were newly installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "scikit-learn", "joblib", "pydantic", "httpx"],
               check=False)
print("Dependencies ready")


In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib, numpy as np

print("Imports OK")


## 3  Train a classifier and save it

We use the Iris dataset (4 features, 3 classes). The saved `.joblib` file is the **model artifact**—the file the API will load at startup.


In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print(f"Test accuracy: {accuracy:.2%}")

# Save the artifact
MODEL_PATH = "/tmp/iris_rf.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")


## 4  Define the FastAPI application

Three key components:
- **Pydantic schema** — validates every incoming request; rejects bad data before it reaches the model.
- **Startup model load** — the model is read from disk once when the server starts, not on every request.
- **`/predict` and `/health`** — predict returns the class name and confidence; health lets load balancers check the service is alive.


In [ ]:
%%writefile /tmp/main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import joblib
import numpy as np

app = FastAPI(title="Iris Classifier API", version="1.0")

# ---------------------------------------------------------------------------
# Schema — Pydantic validates types and raises 422 if fields are missing/wrong
# ---------------------------------------------------------------------------
class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

class PredictResponse(BaseModel):
    prediction: str
    confidence: float

# ---------------------------------------------------------------------------
# Load model at startup — once, not per request
# ---------------------------------------------------------------------------
MODEL_PATH = "/tmp/iris_rf.joblib"
CLASS_NAMES = ["setosa", "versicolor", "virginica"]

@app.on_event("startup")
def load_model():
    global clf
    clf = joblib.load(MODEL_PATH)

# ---------------------------------------------------------------------------
# Endpoints
# ---------------------------------------------------------------------------
@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(req: IrisRequest):
    features = np.array([[req.sepal_length, req.sepal_width,
                           req.petal_length, req.petal_width]])
    proba = clf.predict_proba(features)[0]
    class_idx = int(np.argmax(proba))
    return PredictResponse(
        prediction=CLASS_NAMES[class_idx],
        confidence=round(float(proba[class_idx]), 4)
    )


The file was just written to `/tmp/main.py`. To run the real server:

```bash
uvicorn main:app --host 0.0.0.0 --port 8000 --reload
```

Then test with:

```bash
curl -X POST http://localhost:8000/predict \
     -H 'Content-Type: application/json' \
     -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
```


## 5  Simulate the predict logic in-process

We cannot start a live uvicorn server inside a notebook, but we can import the predict logic directly and call it. This is enough to verify correctness.


In [ ]:
# Simulate what the FastAPI endpoint does, without a live server
import joblib
import numpy as np

CLASS_NAMES = ["setosa", "versicolor", "virginica"]
loaded_clf = joblib.load("/tmp/iris_rf.joblib")

def predict_in_process(sepal_length, sepal_width, petal_length, petal_width):
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    proba = loaded_clf.predict_proba(features)[0]
    class_idx = int(np.argmax(proba))
    return {
        "prediction": CLASS_NAMES[class_idx],
        "confidence": round(float(proba[class_idx]), 4)
    }

# Test with a known setosa sample
result = predict_in_process(5.1, 3.5, 1.4, 0.2)
print("Input: sepal=5.1x3.5  petal=1.4x0.2")
print(f"Prediction : {result['prediction']}")
print(f"Confidence : {result['confidence']:.1%}")

# Test with a virginica sample
result2 = predict_in_process(6.7, 3.0, 5.2, 2.3)
print("\nInput: sepal=6.7x3.0  petal=5.2x2.3")
print(f"Prediction : {result2['prediction']}")
print(f"Confidence : {result2['confidence']:.1%}")


## 6  What happens when the schema is violated

Pydantic enforces the schema before the model ever runs. Below we show what validation errors look like.


In [ ]:
from pydantic import BaseModel, ValidationError

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

# Missing field
try:
    IrisRequest(sepal_length=5.1, sepal_width=3.5, petal_length=1.4)
except ValidationError as e:
    print("Missing field error:")
    print(e)

# Wrong type
try:
    IrisRequest(sepal_length="five", sepal_width=3.5, petal_length=1.4, petal_width=0.2)
except ValidationError as e:
    print("\nWrong type error:")
    print(e)


## Summary

Model serving means wrapping a trained artifact in an HTTP layer so any client can reach it. A FastAPI service has three essential parts:

| Component | What it does |
|---|---|
| Model artifact (`.joblib`) | Stores the trained weights/parameters on disk |
| Pydantic schema | Validates request fields before they touch the model |
| API endpoints (`/predict`, `/health`) | Expose the model over HTTP; health is used by load balancers |

Loading the model at startup (not per request) keeps latency low. The `/health` route lets orchestrators know the service is ready.


## Self-check

Answer from memory first, then verify by re-reading the notebook.

1. **What does Pydantic do in this setup?** Hint: look at the `IrisRequest` class and the validation error cell.
2. **What HTTP status code does FastAPI return when a required field is missing?** Run the schema validation cell and check the error type — then look up what uvicorn would return to a client.
3. **Why do we load the model at server startup rather than inside the `/predict` function?** Think about what happens to latency if the model is loaded on every request.
